# 1.08 - UFO_Parser

This notebook parses for haunted places with UFO and UAP sightings. 

**The functions of this notebooke were merged with 1.05** 


## **Method**
1. Check each description for the following regex patterns using **check_regex** in *parsingFunction.py*
    - Flying Objects | ["ufo"], ["uap"], etc...
    - Ambiguous Objects + Flying Descriptor | ["orb" + "flying"], ["trails" + "air"], etc...
    - Plane Crashes | ["plane" + "crash"], etc...
    - Electronic Malfunctions | ["Electronic" + "jammed"], ["lights", "flickering"], etc.
2. Store each result in a feature column for analysis.


    

In [53]:
# System Path #
import os
import sys 

# Add dsci_550_a1 to base path. Lets you project functions #
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

# Pandas #
import pandas as pd
import json
import re

# Runtime #
import time
from tqdm import tqdm 

# Iterators #
import collections
import ast
import random
from typing import Pattern
from itertools import chain
from collections import Counter

from dsci_550_a1.parsingFunctions import *



In [ ]:
# Output Df
outfile = "../data/processed/haunted_places_features_added.tab"

# Reading CSV
haunted_places_df = pd.read_csv("../data/processed/haunted_places_cleaned.tab", sep = "\t")

# Feature Names
feature_names = ["Apparition_Type"]



In [54]:
## Outfile CSV ##
outfile = "../data/processed/haunted_places_features_added.tab"

## Load Haunted Places Dataset ##
haunted_places_df = pd.read_csv("../data/processed/haunted_places_cleaned.tab", sep = "\t")

## Feature Names ##
feature_names = ["Electronic_Malfunction", "Plane_Crash", "Flying_Object", "Flying_Orb"]

## Keyword Dictionary ##
airborne_keywords = json.load(open("../data/keywords/airborne_keywords.json", "r"))

## regular expressions for nouns ##
flying_object_regex = re.compile(r"\b(" + "|".join(map(re.escape, list(chain(airborne_keywords["Nouns"]['Flying_Objects'])))) + r"s?)\b", re.IGNORECASE)
electronic_equipment_regex = re.compile(r"\b(" + "|".join(map(re.escape, list(chain(airborne_keywords["Nouns"]['Electronic_Equipment'])))) + r"s?)\b", re.IGNORECASE)
ambiguous_object_regex = re.compile(r"\b(" + "|".join(map(re.escape, list(chain(airborne_keywords["Nouns"]['Ambiguous_Objects'])))) + r"s?)\b", re.IGNORECASE)

## regular expressions for descriptors ##
malfunction_descriptor_regex = re.compile(r"\b(" + "|".join(map(re.escape, list(chain(airborne_keywords["Descriptors"]['Malfunction'])))) + r")\b", re.IGNORECASE)
crash_descriptor_regex = re.compile(r"\b(" + "|".join(map(re.escape, list(chain(airborne_keywords["Descriptors"]['Crash'])))) + r")\b", re.IGNORECASE)
flying_descriptor_regex = re.compile(r"\b(" + "|".join(map(re.escape, list(chain(airborne_keywords["Descriptors"]['Flying'])))) + r")\b", re.IGNORECASE)




### Feature Extraction


In [55]:

## Check each event type ##
haunted_places_df["Electronic_Malfunction"] = haunted_places_df["description"].map(lambda x: check_regex(x, electronic_equipment_regex, malfunction_descriptor_regex))

haunted_places_df["Plane_Crash"] = haunted_places_df["description"].map(lambda x: check_regex(x, flying_object_regex, crash_descriptor_regex))

haunted_places_df["Flying_Object"] = haunted_places_df["description"].map(lambda x: check_regex(x, flying_object_regex))

haunted_places_df["Flying_Orb"] = haunted_places_df["description"].map(lambda x: check_regex(x, ambiguous_object_regex, flying_descriptor_regex))


### Exploring Results

In [56]:

## Counting most common Airports ##
feature_counters = {feature : Counter() for feature in feature_names}

for i in haunted_places_df.index:
    for feature in feature_names:
        true_flag, keywords = haunted_places_df.loc[i, feature]
        if true_flag:
            feature_counters[feature][keywords[0]] += 1
            

## Fun Stats ##
for feature in feature_names:
    print("-" * 50, f"Feature Name: [{feature}]", sep = "\n")
    print(f"Total Count : {sum(feature_counters[feature].values())}")
    print(f"Total Coverage : {(sum(feature_counters[feature].values()) / haunted_places_df.shape[0]) *100}%")
    most_common_keywords = "\n".join(f"\t{key} : {value}" for key, value in feature_counters[feature].most_common(10))
    print(f"Most common words flagged for {feature}:", most_common_keywords, "-" * 50, sep = "\n", end = "\n\n")



--------------------------------------------------
Feature Name: [Electronic_Malfunction]
Total Count : 56
Total Coverage : 0.5095077790919843%
Most common words flagged for Electronic_Malfunction:
	car : 36
	electricity : 6
	radio : 4
	computer : 3
	vehicle : 3
	tv : 3
	electronic : 1
--------------------------------------------------

--------------------------------------------------
Feature Name: [Plane_Crash]
Total Count : 16
Total Coverage : 0.1455736511691384%
Most common words flagged for Plane_Crash:
	plane : 14
	helicopter : 2
--------------------------------------------------

--------------------------------------------------
Feature Name: [Flying_Object]
Total Count : 47
Total Coverage : 0.42762260030934396%
Most common words flagged for Flying_Object:
	plane : 20
	craft : 8
	helicopter : 6
	balloon : 4
	ufo : 3
	airplane : 2
	rocket : 1
	shuttle : 1
	chopper : 1
	spaceship : 1
--------------------------------------------------

--------------------------------------------

#### Test Cases

In [57]:
## test cases from haunted_places_cleaned.tab ##
test_cases = [
[4793, "flying_object"],
[9847,  "flying_object"],
[8467,  "flying_object"],
[1405, "flying_object"],
[10664, "flying_object"],

[3449, "electronic_malfunction"],
[1253, "none"],
[1728, "electronic_malfunction"],
[3854, "electronic_malfunction"],


[1470, "plane_crash"],
[7466, "plane_crash"]
]

for test_case in test_cases:
    idx, output = test_case
    print(f"Index: {idx}",f"Description: \n{"\n".join(haunted_places_df.loc[idx, 'description'].split("."))}", sep = "\n", end = "\n")
    
    print(
        "-" * 50,
        f"Target: [{output}]",
        f"Parsed Outputs: \n{haunted_places_df.loc[idx, ["Electronic_Malfunction", "Plane_Crash", "Flying_Object", "Flying_Orb"]]}",
        "-" * 50, 
        sep = "\n",
        end = "\n\n"
    )   



Index: 4793
Description: 
`` gates hell '' nickname cemetery end st
 john road 
 elizabethtown cemetery end saint johns road known gates hell said haunted 
 interesting thing haunted place traveling cemetery one last building see elizabethtown ’ haunted place bethlehem academy 
 couple miles end road surrounded trees overgrowth cemetery contains graves unknown people 1700 ’ 1800 ’ 
 place right outside remains iron stone gate people go party away town 
 many years ago parked night witnesses report watched enormous green orb suddenly suspended right 
 couple long minutes orb shot straight fast sight second 
 others claimed phenomenon hanging cemetery hearing screams seeing shadow people electrical problems cars becoming scared many never returned 

--------------------------------------------------
Target: [flying_object]
Parsed Outputs: 
Electronic_Malfunction                 (False, [])
Plane_Crash                            (False, [])
Flying_Object                          (False, [